# 유휴 공유재산 행정동 연결

## 분석 목적
부산 16개 구·군의 유휴 공유재산 10,102건을 206개 행정동과 연결하고,
앞서 구축한 지역수요 프로필을 각 재산에 결합하여
개별 유휴재산의 지역적 맥락을 분석할 수 있는 기반 데이터를 구축한다.

## 분석 절차
1. 유휴 공유재산 데이터 불러오기 및 구조 확인
2. 주소·좌표 활용 가능성 점검
3. 행정동 연결 방식 결정
4. 행정동 코드 부여
5. 지역수요 프로필 결합
6. 연결 실패·중복·결측 검증

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



In [ ]:
import pandas as pd
import numpy as np

BASE_PATH = "/content/drive/MyDrive/부산_BigData_공모전/"

idle = pd.read_csv(
    BASE_PATH + "02_processed_data/유휴공유재산_16개구군_통합.csv",
    encoding="utf-8-sig"
)

profile = pd.read_csv(
    BASE_PATH + "02_processed_data/지역수요프로필_행정동별.csv",
    encoding="utf-8-sig"
)

print("[유휴 공유재산]")
print("shape:", idle.shape)
print("columns:", idle.columns.tolist())
print("\n결측값:")
print(idle.isna().sum())

print("\n[지역수요 프로필]")
print("shape:", profile.shape)
print("columns:", profile.columns.tolist())

display(idle.head())
display(profile.head())

In [ ]:
from google.colab import userdata
import requests

KAKAO_REST_API_KEY = userdata.get("KAKAO_REST_API_KEY")

test_address = idle.loc[0, "lctnAddr"]

url = "https://dapi.kakao.com/v2/local/search/address.json"

headers = {
    "Authorization": f"KakaoAK {KAKAO_REST_API_KEY}"
}

params = {
    "query": test_address
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=20
)

print("테스트 주소:", test_address)
print("HTTP 상태:", response.status_code)
print("응답 앞부분:", response.text[:500])

if response.status_code == 200:
    data = response.json()
    print("검색 결과 수:", data["meta"]["total_count"])

    if data["documents"]:
        result = data["documents"][0]
        print("검색 주소:", result["address_name"])
        print("경도:", result["x"])
        print("위도:", result["y"])

In [ ]:
# 20건 행정동코드 매칭 테스트
import pandas as pd
import time

profile_codes = set(profile["행정동코드"].astype(str))

test_rows = []

for addr in idle["lctnAddr"].drop_duplicates().head(20):
    response = requests.get(
        "https://dapi.kakao.com/v2/local/search/address.json",
        headers=headers,
        params={"query": addr},
        timeout=20
    )

    if response.status_code == 200:
        data = response.json()

        if data["documents"]:
            doc = data["documents"][0]
            addr_info = doc.get("address") or {}

            h_code = str(addr_info.get("h_code", ""))

            test_rows.append({
                "원본주소": addr,
                "검색주소": doc.get("address_name"),
                "경도": doc.get("x"),
                "위도": doc.get("y"),
                "행정동코드": h_code,
                "행정동코드_프로필매칭": h_code in profile_codes,
                "상태": "OK"
            })

        else:
            test_rows.append({
                "원본주소": addr,
                "검색주소": None,
                "경도": None,
                "위도": None,
                "행정동코드": None,
                "행정동코드_프로필매칭": False,
                "상태": "NO_RESULT"
            })

    else:
        test_rows.append({
            "원본주소": addr,
            "검색주소": None,
            "경도": None,
            "위도": None,
            "행정동코드": None,
            "행정동코드_프로필매칭": False,
            "상태": f"HTTP_{response.status_code}"
        })

    time.sleep(0.1)

test_geo = pd.DataFrame(test_rows)

display(test_geo)

print("\n[테스트 결과]")
print(test_geo["상태"].value_counts(dropna=False))

print(
    "행정동코드 프로필 매칭:",
    test_geo["행정동코드_프로필매칭"].sum(),
    "/",
    len(test_geo)
)

In [ ]:
# 전체 유휴재산 주소 지오코딩
import os
import re
import time
import numpy as np
import pandas as pd
import requests

from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata

KAKAO_REST_API_KEY = userdata.get("KAKAO_REST_API_KEY")

GEOCODE_URL = "https://dapi.kakao.com/v2/local/search/address.json"

headers = {
    "Authorization": f"KakaoAK {KAKAO_REST_API_KEY}"
}

checkpoint_path = (
    BASE_PATH
    + "02_processed_data/유휴공유재산_주소좌표_카카오_중간저장.csv"
)

# 고유 주소만 API 호출
unique_addr = (
    idle[["lctnAddr"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("전체 유휴재산:", len(idle))
print("고유 주소:", len(unique_addr))
print("중복 주소:", len(idle) - len(unique_addr))


def normalize_address(address):
    address = str(address).strip()
    address = re.sub(r"\s+", " ", address)
    address = address.replace(",", " ")
    address = re.sub(r"\s+", " ", address)
    return address.strip()


def geocode_one(address, max_retry=3):

    query = normalize_address(address)

    for attempt in range(max_retry):

        try:
            response = requests.get(
                GEOCODE_URL,
                headers=headers,
                params={"query": query},
                timeout=15
            )

            # 일시적 오류 및 요청 제한
            if response.status_code in [429, 500, 502, 503, 504]:
                time.sleep(1.5 * (attempt + 1))
                continue

            if response.status_code != 200:
                return {
                    "lctnAddr": address,
                    "query_address": query,
                    "x": np.nan,
                    "y": np.nan,
                    "h_code": None,
                    "b_code": None,
                    "api_gugun": None,
                    "api_hdong": None,
                    "match_address": None,
                    "geocode_status": f"HTTP_{response.status_code}"
                }

            data = response.json()

            if not data.get("documents"):
                return {
                    "lctnAddr": address,
                    "query_address": query,
                    "x": np.nan,
                    "y": np.nan,
                    "h_code": None,
                    "b_code": None,
                    "api_gugun": None,
                    "api_hdong": None,
                    "match_address": None,
                    "geocode_status": "NO_RESULT"
                }

            doc = data["documents"][0]
            addr = doc.get("address") or {}

            return {
                "lctnAddr": address,
                "query_address": query,
                "x": float(doc["x"]),
                "y": float(doc["y"]),
                "h_code": str(addr.get("h_code", "")),
                "b_code": str(addr.get("b_code", "")),
                "api_gugun": addr.get("region_2depth_name"),
                "api_hdong": addr.get("region_3depth_h_name"),
                "match_address": doc.get("address_name"),
                "geocode_status": "OK"
            }

        except Exception as e:

            if attempt < max_retry - 1:
                time.sleep(1.5 * (attempt + 1))
                continue

            return {
                "lctnAddr": address,
                "query_address": query,
                "x": np.nan,
                "y": np.nan,
                "h_code": None,
                "b_code": None,
                "api_gugun": None,
                "api_hdong": None,
                "match_address": None,
                "geocode_status": f"ERROR_{type(e).__name__}"
            }


# 이전 작업이 있으면 이어서 시작
if os.path.exists(checkpoint_path):

    geo_cache = pd.read_csv(
        checkpoint_path,
        encoding="utf-8-sig",
        dtype={
            "h_code": str,
            "b_code": str
        }
    )

    print("중간저장 파일 발견:", len(geo_cache))

else:

    geo_cache = pd.DataFrame()


done_addresses = set()

if len(geo_cache) > 0:
    done_addresses = set(
        geo_cache.loc[
            geo_cache["geocode_status"] == "OK",
            "lctnAddr"
        ]
    )


todo = unique_addr[
    ~unique_addr["lctnAddr"].isin(done_addresses)
]["lctnAddr"].tolist()

print("이번 실행 대상:", len(todo))


# 200건씩 처리 후 저장
chunk_size = 200

for start in range(0, len(todo), chunk_size):

    chunk = todo[start:start + chunk_size]
    chunk_results = []

    with ThreadPoolExecutor(max_workers=5) as executor:

        futures = {
            executor.submit(geocode_one, addr): addr
            for addr in chunk
        }

        for future in as_completed(futures):
            chunk_results.append(future.result())

    chunk_df = pd.DataFrame(chunk_results)

    # 기존 동일 주소 결과 제거 후 최신 결과 저장
    if len(geo_cache) > 0:
        geo_cache = geo_cache[
            ~geo_cache["lctnAddr"].isin(chunk_df["lctnAddr"])
        ]

    geo_cache = pd.concat(
        [geo_cache, chunk_df],
        ignore_index=True
    )

    geo_cache.to_csv(
        checkpoint_path,
        index=False,
        encoding="utf-8-sig"
    )

    ok_count = (geo_cache["geocode_status"] == "OK").sum()

    print(
        f"{min(start + chunk_size, len(todo))}/{len(todo)} 처리 | "
        f"누적 OK {ok_count}"
    )


print("\n[전체 지오코딩 결과]")
print(geo_cache["geocode_status"].value_counts(dropna=False))
print("저장:", checkpoint_path)

In [ ]:
# 지오코딩 품질 검증
geo_cache["h_code"] = geo_cache["h_code"].astype("string")

profile["행정동코드"] = profile["행정동코드"].astype(str)

profile_codes = set(profile["행정동코드"])

geo_cache["profile_code_match"] = (
    geo_cache["h_code"].isin(profile_codes)
)

print("[지오코딩 상태]")
print(geo_cache["geocode_status"].value_counts(dropna=False))

print("\n고유 주소:", len(geo_cache))
print("좌표 성공:", geo_cache["x"].notna().sum())
print("행정동코드 성공:", geo_cache["h_code"].notna().sum())

print(
    "206개 프로필 코드와 매칭:",
    geo_cache["profile_code_match"].sum()
)

print(
    "프로필 코드 불일치:",
    (~geo_cache["profile_code_match"]).sum()
)

print("\n[실패/불일치 주소]")
display(
    geo_cache[
        (geo_cache["geocode_status"] != "OK")
        | (~geo_cache["profile_code_match"])
    ][
        [
            "lctnAddr",
            "query_address",
            "match_address",
            "h_code",
            "api_gugun",
            "api_hdong",
            "geocode_status"
        ]
    ].head(100)
)

In [ ]:
# 실패 주소 재검색
import re
import time
import numpy as np
import pandas as pd
import requests

failed_addresses = (
    geo_cache.loc[
        geo_cache["geocode_status"] != "OK",
        "lctnAddr"
    ]
    .drop_duplicates()
    .tolist()
)

print("재검색 대상:", len(failed_addresses))


def make_address_variants(address):
    address = str(address).strip()
    variants = []

    # 원주소 정리
    clean = re.sub(r"\s+", " ", address)
    variants.append(("NORMALIZED", clean))

    # 부산광역시 → 부산
    short_city = clean.replace("부산광역시", "부산")
    variants.append(("SHORT_CITY", short_city))

    # 행정동식 숫자 동명 → 법정동 형태 후보
    # 예: 연산5동 → 연산동
    converted = re.sub(
        r"([가-힣]+?)[1-9]\d*동(?=\s)",
        r"\1동",
        clean
    )

    if converted != clean:
        variants.append(("DONG_NORMALIZED", converted))
        variants.append(
            ("DONG_NORMALIZED_SHORT_CITY",
             converted.replace("부산광역시", "부산"))
        )

    # 중복 제거
    seen = set()
    result = []

    for method, query in variants:
        if query not in seen:
            result.append((method, query))
            seen.add(query)

    return result


def geocode_query(query):
    try:
        r = requests.get(
            "https://dapi.kakao.com/v2/local/search/address.json",
            headers=headers,
            params={"query": query},
            timeout=20
        )

        if r.status_code != 200:
            return None

        data = r.json()

        if not data.get("documents"):
            return None

        doc = data["documents"][0]
        addr = doc.get("address") or {}

        return {
            "x": float(doc["x"]),
            "y": float(doc["y"]),
            "h_code": str(addr.get("h_code", "")),
            "b_code": str(addr.get("b_code", "")),
            "api_gugun": addr.get("region_2depth_name"),
            "api_hdong": addr.get("region_3depth_h_name"),
            "match_address": doc.get("address_name")
        }

    except Exception:
        return None


recovered = []

for i, address in enumerate(failed_addresses, start=1):

    original_gugun = (
        idle.loc[idle["lctnAddr"] == address, "gugun"]
        .iloc[0]
    )

    found = False

    for method, query in make_address_variants(address):

        result = geocode_query(query)

        if result is None:
            continue

        # 원자료 구군과 일치할 때만 복구 인정
        if result["api_gugun"] != original_gugun:
            continue

        recovered.append({
            "lctnAddr": address,
            "recovery_method": method,
            "query_address_recovery": query,
            **result
        })

        found = True
        break

    if i % 20 == 0:
        print(f"{i}/{len(failed_addresses)} 재검색 완료")

    time.sleep(0.08)


recovered_df = pd.DataFrame(recovered)

print("\n복구 성공:", len(recovered_df))
print("여전히 실패:", len(failed_addresses) - len(recovered_df))

if len(recovered_df) > 0:
    print("\n[복구 방식]")
    print(recovered_df["recovery_method"].value_counts())

    display(
        recovered_df[
            [
                "lctnAddr",
                "query_address_recovery",
                "match_address",
                "h_code",
                "api_hdong",
                "recovery_method"
            ]
        ]
    )

In [ ]:
# 재검색 성공 결과 반영
if len(recovered_df) > 0:

    if "recovery_method" not in geo_cache.columns:
        geo_cache["recovery_method"] = None

    for _, row in recovered_df.iterrows():

        mask = geo_cache["lctnAddr"] == row["lctnAddr"]

        geo_cache.loc[mask, "x"] = row["x"]
        geo_cache.loc[mask, "y"] = row["y"]
        geo_cache.loc[mask, "h_code"] = row["h_code"]
        geo_cache.loc[mask, "b_code"] = row["b_code"]
        geo_cache.loc[mask, "api_gugun"] = row["api_gugun"]
        geo_cache.loc[mask, "api_hdong"] = row["api_hdong"]
        geo_cache.loc[mask, "match_address"] = row["match_address"]
        geo_cache.loc[mask, "geocode_status"] = "OK_RECOVERED"
        geo_cache.loc[mask, "recovery_method"] = row["recovery_method"]


geo_cache.to_csv(
    checkpoint_path,
    index=False,
    encoding="utf-8-sig"
)

print(geo_cache["geocode_status"].value_counts(dropna=False))

In [ ]:
# 미복구 주소 구조 확인
remaining = geo_cache[
    ~geo_cache["geocode_status"].isin(["OK", "OK_RECOVERED"])
][["lctnAddr", "geocode_status"]].copy()

remaining = remaining.merge(
    idle[["lctnAddr", "gugun"]].drop_duplicates(),
    on="lctnAddr",
    how="left"
)

print("최종 미복구 주소:", len(remaining))

print("\n[구군별 미복구]")
print(
    remaining["gugun"]
    .value_counts()
)

display(remaining)

In [ ]:
# 미복구 주소의 본번(parent lot)으로 재검색
remaining_addresses = remaining["lctnAddr"].tolist()

def make_parent_lot_address(address):
    address = str(address).strip()

    # 끝부분이 123-45 형태일 때 자번 제거
    parent = re.sub(r"(\d+)-\d+$", r"\1", address)

    if parent == address:
        return None

    return parent


parent_recovered = []

for i, address in enumerate(remaining_addresses, start=1):

    parent_query = make_parent_lot_address(address)

    if parent_query is None:
        continue

    original_gugun = (
        idle.loc[idle["lctnAddr"] == address, "gugun"]
        .iloc[0]
    )

    result = geocode_query(parent_query)

    if result is None:
        continue

    # 구군 + 행정동코드 모두 검증
    if (
        result["api_gugun"] == original_gugun
        and result["h_code"] in profile_codes
    ):
        parent_recovered.append({
            "lctnAddr": address,
            "query_address_recovery": parent_query,
            "recovery_method": "PARENT_LOT_APPROX",
            **result
        })

    if i % 20 == 0:
        print(f"{i}/{len(remaining_addresses)} 확인")

    time.sleep(0.08)


parent_recovered_df = pd.DataFrame(parent_recovered)

print("본번 검색 복구:", len(parent_recovered_df))
print(
    "남은 미복구:",
    len(remaining_addresses) - len(parent_recovered_df)
)

if len(parent_recovered_df) > 0:
    display(
        parent_recovered_df[
            [
                "lctnAddr",
                "query_address_recovery",
                "match_address",
                "h_code",
                "api_hdong"
            ]
        ]
    )

In [ ]:
# 본번 검색 성공 결과 반영
if len(parent_recovered_df) > 0:

    for _, row in parent_recovered_df.iterrows():

        mask = geo_cache["lctnAddr"] == row["lctnAddr"]

        geo_cache.loc[mask, "x"] = row["x"]
        geo_cache.loc[mask, "y"] = row["y"]
        geo_cache.loc[mask, "h_code"] = row["h_code"]
        geo_cache.loc[mask, "b_code"] = row["b_code"]
        geo_cache.loc[mask, "api_gugun"] = row["api_gugun"]
        geo_cache.loc[mask, "api_hdong"] = row["api_hdong"]
        geo_cache.loc[mask, "match_address"] = row["match_address"]
        geo_cache.loc[mask, "geocode_status"] = "OK_PARENT_LOT"
        geo_cache.loc[mask, "recovery_method"] = "PARENT_LOT_APPROX"

geo_cache.to_csv(
    checkpoint_path,
    index=False,
    encoding="utf-8-sig"
)

print(geo_cache["geocode_status"].value_counts(dropna=False))

In [ ]:
# 법정동 → 행정동 관계 검증
geo_check = geo_cache.merge(
    idle[["lctnAddr", "gugun"]].drop_duplicates(),
    on="lctnAddr",
    how="left"
)

geo_check["법정동명_주소"] = (
    geo_check["lctnAddr"]
    .str.extract(r"부산광역시\s+\S+\s+(\S+)", expand=False)
)

success_geo = geo_check[
    geo_check["geocode_status"].isin(
        ["OK", "OK_RECOVERED", "OK_PARENT_LOT"]
    )
].copy()

legal_to_admin = (
    success_geo
    .groupby(["gugun", "법정동명_주소"])
    .agg(
        행정동코드수=("h_code", "nunique"),
        행정동수=("api_hdong", "nunique"),
        대표행정동코드=("h_code", "first"),
        대표행정동=("api_hdong", "first")
    )
    .reset_index()
)

still_failed = geo_check[
    ~geo_check["geocode_status"].isin(
        ["OK", "OK_RECOVERED", "OK_PARENT_LOT"]
    )
].copy()

failed_mapping_check = still_failed.merge(
    legal_to_admin,
    on=["gugun", "법정동명_주소"],
    how="left"
)

print("현재 미복구:", len(failed_mapping_check))

print("\n[법정동이 단 하나의 행정동에만 대응]")
print(
    (failed_mapping_check["행정동코드수"] == 1).sum()
)

print("\n[복수 행정동 또는 판단 불가]")
print(
    (failed_mapping_check["행정동코드수"] != 1).sum()
)

display(
    failed_mapping_check[
        [
            "lctnAddr",
            "gugun",
            "법정동명_주소",
            "행정동코드수",
            "대표행정동",
            "대표행정동코드"
        ]
    ]
)

In [ ]:
# 법정동이 하나의 행정동에만 대응하는 미복구 주소 보완
safe_mapping = failed_mapping_check[
    failed_mapping_check["행정동코드수"] == 1
].copy()

print("안전 보완 대상:", len(safe_mapping))

display(
    safe_mapping[
        [
            "lctnAddr",
            "gugun",
            "법정동명_주소",
            "대표행정동",
            "대표행정동코드"
        ]
    ]
)

if "recovery_method" not in geo_cache.columns:
    geo_cache["recovery_method"] = None

for _, row in safe_mapping.iterrows():

    mask = geo_cache["lctnAddr"] == row["lctnAddr"]

    geo_cache.loc[mask, "h_code"] = str(row["대표행정동코드"])
    geo_cache.loc[mask, "api_gugun"] = row["gugun"]
    geo_cache.loc[mask, "api_hdong"] = row["대표행정동"]

    # 좌표는 추정하지 않음
    geo_cache.loc[mask, "geocode_status"] = "OK_LEGAL_DONG"
    geo_cache.loc[mask, "recovery_method"] = "LEGAL_DONG_1TO1"

print("\n[최종 상태]")
print(geo_cache["geocode_status"].value_counts(dropna=False))

In [ ]:
# 위치 매칭 방식 및 정확도 구분
status_to_method = {
    "OK": "EXACT_ADDRESS",
    "OK_RECOVERED": "NORMALIZED_ADDRESS",
    "OK_PARENT_LOT": "PARENT_LOT_APPROX",
    "OK_LEGAL_DONG": "LEGAL_DONG_INFERRED",
    "NO_RESULT": "UNRESOLVED"
}

geo_cache["location_match_method"] = (
    geo_cache["geocode_status"]
    .map(status_to_method)
)

geo_cache["coordinate_available"] = (
    geo_cache["x"].notna()
    & geo_cache["y"].notna()
)

print("[매칭 방식]")
print(geo_cache["location_match_method"].value_counts(dropna=False))

print("\n좌표 확보:", geo_cache["coordinate_available"].sum())
print("행정동코드 확보:", geo_cache["h_code"].notna().sum())

In [ ]:
# 끝까지 위치를 확정하지 못한 주소 보존
unresolved_geo = geo_cache[
    geo_cache["geocode_status"] == "NO_RESULT"
].copy()

unresolved_geo = unresolved_geo.merge(
    idle[
        [
            "lctnAddr",
            "gugun",
            "zoneArea",
            "land_type",
            "data_source"
        ]
    ].drop_duplicates(subset=["lctnAddr"]),
    on="lctnAddr",
    how="left"
)

unresolved_path = (
    BASE_PATH
    + "02_processed_data/유휴공유재산_위치미매칭_60건.csv"
)

unresolved_geo.to_csv(
    unresolved_path,
    index=False,
    encoding="utf-8-sig"
)

print("미매칭:", len(unresolved_geo))
print("저장:", unresolved_path)

print("\n[구군별]")
print(unresolved_geo["gugun"].value_counts())

In [ ]:
# 주소별 위치정보를 원본 10,102건에 결합
geo_cols = [
    "lctnAddr",
    "x",
    "y",
    "h_code",
    "b_code",
    "api_gugun",
    "api_hdong",
    "match_address",
    "geocode_status",
    "recovery_method",
    "location_match_method",
    "coordinate_available"
]

print("주소별 위치정보 중복:",
      geo_cache["lctnAddr"].duplicated().sum())

idle_geo = idle.merge(
    geo_cache[geo_cols],
    on="lctnAddr",
    how="left",
    validate="many_to_one"
)

print("\n원본 행:", len(idle))
print("결합 후:", len(idle_geo))
print("행 수 변화:", len(idle_geo) - len(idle))

print("좌표 연결:",
      idle_geo["coordinate_available"].sum())

print("행정동코드 연결:",
      idle_geo["h_code"].notna().sum())

print("미연결:",
      idle_geo["h_code"].isna().sum())

In [ ]:
# API로 위치가 확인된 자료의 구군 교차검증
api_located = idle_geo["geocode_status"].isin(
    ["OK", "OK_RECOVERED", "OK_PARENT_LOT"]
)

idle_geo["구군_일치"] = pd.NA

idle_geo.loc[api_located, "구군_일치"] = (
    idle_geo.loc[api_located, "gugun"].astype(str).str.strip()
    ==
    idle_geo.loc[api_located, "api_gugun"].astype(str).str.strip()
)

print("API 위치확인:", api_located.sum())

print(
    "구군 일치:",
    (idle_geo.loc[api_located, "구군_일치"] == True).sum()
)

print(
    "구군 불일치:",
    (idle_geo.loc[api_located, "구군_일치"] == False).sum()
)

display(
    idle_geo[
        api_located
        & (idle_geo["구군_일치"] == False)
    ][
        [
            "gugun",
            "lctnAddr",
            "match_address",
            "api_gugun",
            "api_hdong",
            "h_code"
        ]
    ]
)

In [ ]:
# 행정동코드 형식 통일
idle_geo["h_code"] = idle_geo["h_code"].astype("string")

profile_merge = profile.copy()
profile_merge["행정동코드"] = (
    profile_merge["행정동코드"]
    .astype("string")
)

# 지역수요 프로필 결합
idle_analysis = idle_geo.merge(
    profile_merge,
    left_on="h_code",
    right_on="행정동코드",
    how="left",
    validate="many_to_one"
)

print("결합 전:", len(idle_geo))
print("결합 후:", len(idle_analysis))
print("행 수 변화:", len(idle_analysis) - len(idle_geo))

print(
    "\n지역수요 연결 성공:",
    idle_analysis["행정동명"].notna().sum()
)

print(
    "지역수요 연결 실패:",
    idle_analysis["행정동명"].isna().sum()
)

print(
    "연결된 행정동 수:",
    idle_analysis["행정동코드"].nunique()
)

print("\n[미연결 상태]")
print(
    idle_analysis.loc[
        idle_analysis["행정동명"].isna(),
        "geocode_status"
    ].value_counts(dropna=False)
)

In [ ]:
# 최종 분석용 데이터 저장
final_path = (
    BASE_PATH
    + "02_processed_data/유휴공유재산_지역수요결합.csv"
)

idle_analysis.to_csv(
    final_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", final_path)
print("최종 shape:", idle_analysis.shape)

In [ ]:
geo_final_path = (
    BASE_PATH
    + "02_processed_data/유휴공유재산_주소위치정보.csv"
)

geo_cache.to_csv(
    geo_final_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", geo_final_path)

# ✅ 유휴 공유재산 위치정보 구축 결과

## 최종 결과

부산 유휴 공유재산 10,102건의 지번주소를 대상으로
카카오 Local API를 활용하여 좌표 및 행정동 정보를 구축하였다.

### 위치정보 연결 결과

- 전체 유휴 공유재산: **10,102건**
- 정확 주소 매칭: **10,022건**
- 주소 정규화 후 복구: **1건**
- 본번 기준 위치 복구: **14건**
- 법정동-행정동 1:1 관계를 이용한 행정동코드 보완: **5건**
- 최종 행정동코드 확보: **10,042건**
- 최종 미매칭: **60건**

행정동코드가 확보된 10,042건은
앞서 구축한 부산 206개 행정동 지역수요 프로필과 정상적으로 결합되었다.

## 검증

- 원본 결합 전후 행 수: **10,102건 유지**
- 중복으로 인한 행 증가: 없음
- API 기반 위치확인 자료의 구·군 불일치: **0건**
- 지역수요 프로필 연결 성공: **10,042건**
- 지역수요 프로필 연결 실패: **60건**

## 미매칭 자료 처리

최종 미매칭 60건은 임의의 행정동을 부여하거나 삭제하지 않았다.

따라서 이후 분석에서는

- 유휴재산 자체의 면적·지목·구·군 분석: **10,102건 전체 사용**
- 행정동 지역수요가 필요한 분석: **10,042건 사용**
- 좌표가 필요한 공간분석: 위치 매칭 방식과 좌표 정확도를 구분하여 사용

한다.

특히 본번 기준으로 복구된 자료는 정확한 자번 위치가 아니라
해당 본번의 대표 위치이므로 정밀 거리분석 시 구분하여 해석한다.

## 최종 산출파일

- `유휴공유재산_주소위치정보.csv`
- `유휴공유재산_위치미매칭_60건.csv`
- `유휴공유재산_지역수요결합.csv`